# Figure: cis-eQTL variant-effect benchmark (ROC / PR + AUC by distance)

This figure benchmarks Shorkie (and the `Shorkie_LM` and `Shorkie_Random_Init` ablations) against the DREAM MPRA models (DREAM-Atten, DREAM-CNN, DREAM-RNN) on the task of classifying cis-eQTL variants vs. matched negatives, using the absolute logSED variant-effect score. Panel 1 shows ROC and precision-recall curves aggregated as **mean +/- SEM over the 4 matched negative sets** (interpolated mean curve with a shaded +/-1 SEM band). Panel 2 stratifies AUROC / AUPRC by the variant-to-TSS distance bin to show how performance decays with distance.

**Reproduces:** the eQTL ROC/PR ensemble curves and the AUC-by-distance panels for one representative dataset (Caudal et al.).

**Upstream:** the per-negative-set score TSVs are produced by the eQTL variant-scoring + parsing stage under `scripts/04_analysis/shorkie/eqtl/` (e.g. `scripts/04_analysis/shorkie/eqtl/3_visualization/0_parse_eqtl_res.py`), and the DREAM model prediction TSVs by the MPRA-model evaluation under `scripts/04_analysis/shorkie/mpra/`. These intermediates are **not** in the released data, so this notebook is not end-to-end runnable; it reads them via `config.path('results.eqtl_scores')` and `config.path('results.mpra_eval')`.

**Requires:** the `yeast_ml` conda env with this package installed (`pip install -e .`). No GPU needed (it only reads TSVs and plots).

**Source script:** ported from `scripts/04_analysis/shorkie/eqtl/3_visualization/1_roc_pr_shorkie_fold.py` and `scripts/04_analysis/shorkie/eqtl/3_visualization/2_AUROC_AUPRC_by_dsitance.py`. The original scripts read paths from an undefined module-level `args.root_dir`; here every path is resolved through `shorkie.config`.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
)

from shorkie import config

SEED = 42
np.random.seed(SEED)

In [ ]:
# Path resolution via shorkie.config (no hardcoded absolute paths).
# results.eqtl_scores  -> dir containing negset_{1..4}/{exp}_Shorkie*_scores.tsv
# results.mpra_eval    -> dir containing DREAM_* model subdirs with prediction TSVs
config.load()
EQTL_SCORES_ROOT = config.path('results.eqtl_scores')
MPRA_EVAL_ROOT = config.path('results.mpra_eval')
print('eQTL scores root:', EQTL_SCORES_ROOT)
print('MPRA eval root  :', MPRA_EVAL_ROOT)

# Representative dataset for this figure
EXP = 'caudal_etal'
EXP_TITLE = 'Caudal et al.'
NEG_SETS = [1, 2, 3, 4]

MPRA_MODELS = ['DREAM_Atten', 'DREAM_CNN', 'DREAM_RNN']
MPRA_NAME_MAP = {'DREAM_Atten': 'DREAM-Atten', 'DREAM_CNN': 'DREAM-CNN', 'DREAM_RNN': 'DREAM-RNN'}

MODEL_COLORS = {
    'Shorkie':             '#2196F3',
    'Shorkie_LM':          '#E91E63',
    'Shorkie_Random_Init': '#FF9800',
    'DREAM-Atten':         '#4CAF50',
    'DREAM-CNN':           '#9C27B0',
    'DREAM-RNN':           '#795548',
}


def get_color(col):
    return MODEL_COLORS.get(col, 'gray')


def get_style(col):
    # solid heavier line for Shorkie variants, dash-dot for DREAM baselines
    return ('-', 2.0) if 'Shorkie' in col else ('-.', 1.5)

In [ ]:
# --- Loaders (ported from the source scripts; args.root_dir -> config paths) ---

def process_shorkie(exp, neg_set):
    """Load Shorkie / Shorkie_LM / Shorkie_Random_Init scores for one neg set.
    Score column = absolute variant-effect (|logSED_agg| for Shorkie & Random_Init,
    |LLR| for the LM). Includes 'distance' (variant-to-TSS) for stratification."""
    base_dir = os.path.join(str(EQTL_SCORES_ROOT), f'negset_{neg_set}')

    df_orig = pd.read_csv(os.path.join(base_dir, f'{exp}_Shorkie_scores.tsv'),
                          sep='\t', usecols=['Position_Gene', 'logSED_agg', 'label', 'distance'])
    df_orig['Position_Gene'] = df_orig['Position_Gene'].astype(str).str.strip()
    df_orig = df_orig.drop_duplicates(['Position_Gene'])
    df_orig['Shorkie'] = df_orig['logSED_agg'].abs()
    df_orig = df_orig.drop(columns=['logSED_agg'])

    df_lm = pd.read_csv(os.path.join(base_dir, f'{exp}_Shorkie_LM_scores.tsv'),
                        sep='\t', usecols=['Position_Gene', 'LLR', 'label'])
    df_lm['Position_Gene'] = df_lm['Position_Gene'].astype(str).str.strip()
    df_lm = df_lm.drop_duplicates(['Position_Gene'])
    df_lm['Shorkie_LM'] = df_lm['LLR'].abs()
    df_lm = df_lm.drop(columns=['LLR', 'label'])

    df_ri = pd.read_csv(os.path.join(base_dir, f'{exp}_Shorkie_Random_Init_scores.tsv'),
                        sep='\t', usecols=['Position_Gene', 'logSED_agg'])
    df_ri['Position_Gene'] = df_ri['Position_Gene'].astype(str).str.strip()
    df_ri = df_ri.drop_duplicates(['Position_Gene'])
    df_ri['Shorkie_Random_Init'] = df_ri['logSED_agg'].abs()
    df_ri = df_ri.drop(columns=['logSED_agg'])

    merged = pd.merge(df_orig, df_lm, on='Position_Gene', how='inner')
    merged = pd.merge(merged, df_ri, on='Position_Gene', how='inner')
    merged['label'] = merged['label'].astype(int)
    return merged


def process_mpra(model, neg_set):
    """Load DREAM model |logSED| predictions; pos=label 1, neg=label 0."""
    model_dir = os.path.join(str(MPRA_EVAL_ROOT), model)
    pos = pd.read_csv(os.path.join(model_dir, 'final_pos_predictions.tsv'),
                      sep='\t', usecols=['Position_Gene', 'logSED'])
    neg = pd.read_csv(os.path.join(model_dir, f'final_neg_predictions_{neg_set}.tsv'),
                      sep='\t', usecols=['Position_Gene', 'logSED'])
    pos['label'], neg['label'] = 1, 0
    for df in (pos, neg):
        df['Position_Gene'] = df['Position_Gene'].astype(str).str.strip()
        df.rename(columns={'logSED': 'score'}, inplace=True)
        df['score'] = df['score'].abs()
    return pd.concat([pos, neg], ignore_index=True)[['Position_Gene', 'score', 'label']]


def load_combined(exp, neg_set):
    """Merge all model scores for one experiment + one negative set on Position_Gene."""
    combined = process_shorkie(exp, neg_set)
    for model in MPRA_MODELS:
        name = MPRA_NAME_MAP[model]
        mpra = process_mpra(model, neg_set).rename(columns={'score': name})
        combined = combined.merge(mpra[['Position_Gene', name]], on='Position_Gene', how='inner')
    return combined


SCORE_COLS = ['Shorkie', 'Shorkie_LM', 'Shorkie_Random_Init'] + [MPRA_NAME_MAP[m] for m in MPRA_MODELS]

In [ ]:
# --- Panel 1: ensemble ROC / PR over the 4 negative sets (mean +/- SEM) ---
COMMON_FPR = np.linspace(0, 1, 200)
COMMON_REC = np.linspace(0, 1, 200)

roc_data = {c: {'tprs': [], 'aucs': []} for c in SCORE_COLS}
pr_data = {c: {'precs': [], 'auprcs': []} for c in SCORE_COLS}

for ns in NEG_SETS:
    df = load_combined(EXP, ns).dropna(subset=SCORE_COLS)
    for col in SCORE_COLS:
        valid = df[df[col] > 0]
        if len(valid) < 10 or valid['label'].nunique() < 2:
            continue
        # ROC: interpolate TPR onto a common FPR grid
        fpr, tpr, _ = roc_curve(valid['label'], valid[col])
        tpr_interp = np.interp(COMMON_FPR, fpr, tpr)
        tpr_interp[0] = 0.0
        roc_data[col]['tprs'].append(tpr_interp)
        roc_data[col]['aucs'].append(auc(fpr, tpr))
        # PR: precision_recall_curve returns decreasing recall -> flip to increasing for interp
        prec, rec, _ = precision_recall_curve(valid['label'], valid[col])
        prec_interp = np.interp(COMMON_REC, rec[::-1], prec[::-1])
        pr_data[col]['precs'].append(prec_interp)
        pr_data[col]['auprcs'].append(average_precision_score(valid['label'], valid[col]))

print('curves collected per model (neg sets used):',
      {c: len(roc_data[c]['aucs']) for c in SCORE_COLS})

In [ ]:
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(11, 5))

# ROC ensemble
ax_roc.plot([0, 1], [0, 1], '--', alpha=0.5, color='gray')
for col in SCORE_COLS:
    tprs, aucs_list = roc_data[col]['tprs'], roc_data[col]['aucs']
    if len(tprs) < 2:
        continue
    mean_tpr = np.mean(tprs, axis=0)
    sem_tpr = sp_stats.sem(tprs, axis=0)
    ls, lw = get_style(col)
    color = get_color(col)
    ax_roc.plot(COMMON_FPR, mean_tpr, linestyle=ls, lw=lw, color=color,
                label=f'{col} ({np.mean(aucs_list):.3f}\u00b1{sp_stats.sem(aucs_list):.3f})')
    ax_roc.fill_between(COMMON_FPR, np.clip(mean_tpr - sem_tpr, 0, 1),
                        np.clip(mean_tpr + sem_tpr, 0, 1), alpha=0.15, color=color)
ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
ax_roc.set_title(f'ROC ensemble ({EXP_TITLE}, 4 neg sets)')
ax_roc.legend(loc='lower right', fontsize=9)

# PR ensemble
for col in SCORE_COLS:
    precs, auprcs_list = pr_data[col]['precs'], pr_data[col]['auprcs']
    if len(precs) < 2:
        continue
    mean_prec = np.mean(precs, axis=0)
    sem_prec = sp_stats.sem(precs, axis=0)
    ls, lw = get_style(col)
    color = get_color(col)
    ax_pr.plot(COMMON_REC, mean_prec, linestyle=ls, lw=lw, color=color,
               label=f'{col} ({np.mean(auprcs_list):.3f}\u00b1{sp_stats.sem(auprcs_list):.3f})')
    ax_pr.fill_between(COMMON_REC, np.clip(mean_prec - sem_prec, 0, 1),
                       np.clip(mean_prec + sem_prec, 0, 1), alpha=0.15, color=color)
ax_pr.set_ylim(0.45, 1.05)
ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_pr.set_title(f'PR ensemble ({EXP_TITLE}, 4 neg sets)')
ax_pr.legend(loc='best', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# --- Panel 2: AUROC / AUPRC stratified by variant-to-TSS distance bin ---
# (ported from 2_AUROC_AUPRC_by_dsitance.py; per neg-set metrics aggregated by seaborn regplot)

# Caudal et al. distance binning
bins = [0, 1000, 2000, 3000, 4000, 5000]
bin_labels = ['0-1kb', '1kb-2kb', '2kb-3kb', '3k-4kb', '4k-5kb']
bin_map = dict(zip(bin_labels, bins[1:]))


def metrics_by_bin(sub, model_name):
    """AUROC/AUPRC per distance bin for one (model, neg-set)."""
    out = []
    sub = sub.dropna(subset=['score', 'distance']).copy()
    sub['distance_bin'] = pd.cut(sub['distance'], bins=bins, labels=bin_labels)
    for b in bin_labels:
        seg = sub[(sub.distance_bin == b) & (sub.score > 0)]
        if len(seg) < 10 or seg['label'].nunique() < 2:
            continue
        y, s = seg['label'], seg['score']
        fpr, tpr, _ = roc_curve(y, s)
        out.append({'model': model_name, 'distance': bin_map[b],
                    'n_pos': int((y == 1).sum()), 'n_neg': int((y == 0).sum()),
                    'roc_auc': auc(fpr, tpr), 'pr_auc': average_precision_score(y, s)})
    return out

rows = []
for ns in NEG_SETS:
    df = load_combined(EXP, ns).dropna(subset=SCORE_COLS)
    for col in SCORE_COLS:
        temp = df[['Position_Gene', 'label', 'distance']].copy()
        temp['score'] = df[col]
        rows += metrics_by_bin(temp, col)

dist_df = pd.DataFrame(rows)
print(dist_df.head())

In [ ]:
# x-tick labels carry pos/neg counts per bin
counts = dist_df.groupby('distance')[['n_pos', 'n_neg']].first()
xtick_labels = []
for lbl in bin_labels:
    d = bin_map[lbl]
    n_p, n_n = (counts.loc[d, 'n_pos'], counts.loc[d, 'n_neg']) if d in counts.index else (0, 0)
    xtick_labels.append(f'{lbl}\nPos: {n_p}\nNeg: {n_n}')

palette = {m: MODEL_COLORS.get(m, 'gray') for m in SCORE_COLS}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
for ax, metric, ylab in [(ax1, 'roc_auc', 'AUROC'), (ax2, 'pr_auc', 'AUPRC')]:
    sns.scatterplot(data=dist_df, x='distance', y=metric, hue='model', style='model',
                    marker='o', s=60, edgecolor='w', palette=palette, legend=False, ax=ax)
    for base in dist_df.model.unique():
        sns.regplot(data=dist_df[dist_df.model == base], x='distance', y=metric,
                    scatter=False, label=base, color=MODEL_COLORS.get(base, 'gray'), ax=ax)
    ax.set_xticks(list(bin_map.values()))
    ax.set_xticklabels(xtick_labels, fontsize=9)
    ax.set_xlabel('TSS distance bin'); ax.set_ylabel(ylab)
    ax.set_title(f'{EXP_TITLE}: {ylab} by distance bin')
    ax.legend(ncol=2, fontsize=8, frameon=False)
plt.tight_layout()
plt.show()